# Cleaning Eniac's Data

## Imports and Data Reads

In [3]:
import pandas as pd

In [4]:
# orders.csv
url = "https://drive.google.com/file/d/1J-XJS9NS6fXH-dA9KEurAJ-MqNhVJqEE/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders_df = pd.read_csv(path)

# orderlines.csv
url = "https://drive.google.com/file/d/1LyNo5EZIB0LUJvkrWzsgRsGuVwvnSMBQ/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines_df = pd.read_csv(path)

# products.csv
url = "https://drive.google.com/file/d/1mbJgcLmn8OT-V3yJLgh3D5J50RV411sP/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products_df = pd.read_csv(path)

# brands.csv
url = "https://drive.google.com/file/d/1itjfSCQVkxsJKSuiUNvJ8HJFKENkFZt6/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
brands_df = pd.read_csv(path)

## Orders

In [5]:
# drop rows with missing total_paid
orders_df = orders_df.dropna(axis=0).copy()

In [6]:
# convert created_date to datetime format
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

## Orderlines

In [7]:
# convert date to datetime format
orderlines_df["date"] = pd.to_datetime(orderlines_df["date"])

In [8]:
# remove entries that limit conversion to numeric
# Boolean mask to find the orders that contain a price with multiple decimal points
multiple_decimal_mask = orderlines_df['unit_price'].str.count(r"\.") > 1

# get the rows with corrupted data entry (multiple decimal places)
corrupted_order_ids = orderlines_df.loc[multiple_decimal_mask, "id_order"]

# keep only the rows that do not have multiple decimal points
orderlines_df = orderlines_df.loc[~orderlines_df['id_order'].isin(corrupted_order_ids)]

# bonvert unit_price to numerical format
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

In [9]:
# drop columns not needed for analysis
orderlines_df = orderlines_df.drop(["product_id", "id"], axis=1)

In [10]:
# rename id_order for easier merging
orderlines_df = orderlines_df.rename({"id_order": "order_id"}, axis=1)

## Products

In [11]:
# drop duplicate rows
products_df = products_df.drop_duplicates().copy()

In [12]:
# fill in missing `desc` values with corresponding name
products_df["desc"] = products_df["desc"].fillna(products_df["name"])

##Brands

- requires no cleaning

In [13]:
# drop rows missing `price`
products_df = products_df.dropna(subset="price").copy()

# drop rows with corrupted prices (double decimal)
products_df = products_df.loc[~(products_df["price"].str.count(r"\.") > 1)].copy()

# convert price column to numeric format
products_df["price"] = pd.to_numeric(products_df["price"])

In [14]:
# drop unneeded columns
products_df = products_df.drop(["promo_price", "in_stock"], axis=1)

## Data Exports

In [18]:
#from google.colab import files
orders_df.to_csv("orders_cl.csv", index=False)
files.download("orders_cl.csv")

orderlines_df.to_csv("orderlines_cl.csv", index=False)
files.download("orderlines_cl.csv")

products_df.to_csv("products_cl.csv", index=False)
files.download("products_cl.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>